# 1.1 · What MCP is and why it exists

**什么是 MCP，为什么需要它**

> **After this lesson you can:** Explain the integration problem MCP solves, name its participants and primitives, and connect a client to a first server.

Official reading: [What is MCP?](https://modelcontextprotocol.io/docs/2026-07-28/getting-started/intro) · [Specification overview](https://modelcontextprotocol.io/specification/2026-07-28/index) · [Design principles](https://modelcontextprotocol.io/community/design-principles)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. The N × M problem in numbers
Before any code that talks MCP, make the economic argument concrete. Change the numbers and watch the gap grow.

In [ ]:
def integrations(apps: int, systems: int) -> tuple[int, int]:
    """Connectors to write without a standard vs. with MCP."""
    return apps * systems, apps + systems


for apps, systems in [(3, 5), (10, 50), (40, 2000)]:
    without, with_mcp = integrations(apps, systems)
    print(f"{apps:>3} apps × {systems:>4} systems -> {without:>6} bespoke connectors | "
          f"{with_mcp:>5} MCP components  ({without / with_mcp:.0f}x less work)")

### 2. A first server: one tool, one resource, one prompt
The official Python SDK (`mcp` 2.x) gives you `MCPServer`. Decorators turn ordinary Python functions into the three primitives: type hints become the JSON Schema the model sees, docstrings become descriptions.

In [ ]:
import json

from mcp import Client
from mcp.server import MCPServer

notes = {"mcp": "MCP is USB-C for AI applications.", "lsp": "LSP inspired MCP's design."}

server = MCPServer("notes")


@server.tool()
def search_notes(query: str) -> list[str]:
    """Return the ids of notes whose text contains the query (case-insensitive)."""
    return [nid for nid, text in notes.items() if query.lower() in text.lower()]


@server.resource("notes://{note_id}")
def read_note(note_id: str) -> str:
    """The full text of one note."""
    return notes[note_id]


@server.prompt()
def explain(topic: str) -> str:
    """Ask the model to explain a topic to a beginner."""
    return f"Explain {topic} to a new AI engineer in three sentences."


print("server defined:", server.name)

### 3. Connect a client and discover what the server offers
`Client(server)` connects in-process — perfect for learning and tests. The client *discovers* capabilities at runtime: nothing about the tools is hard-coded on the client side.

In [ ]:
async with Client(server) as client:
    print("protocol version:", client.protocol_version)
    tools = (await client.list_tools()).tools
    templates = (await client.list_resource_templates()).resource_templates
    prompts = (await client.list_prompts()).prompts

    for t in tools:
        print("TOOL     ", t.name, "-", t.description)
        print("          input schema:", json.dumps(t.input_schema))
    for r in templates:
        print("RESOURCE ", r.uri_template, "-", r.description)
    for p in prompts:
        print("PROMPT   ", p.name, "- args:", [a.name for a in p.arguments or []])

### 4. Use all three primitives
Notice the three different *controllers*: the model would call the tool, the application reads the resource, the user picks the prompt.

In [ ]:
async with Client(server) as client:
    hit = await client.call_tool("search_notes", {"query": "usb"})      # model-controlled
    print("tools/call      ->", hit.structured_content, "| isError:", hit.is_error)

    doc = await client.read_resource("notes://mcp")                      # application-controlled
    print("resources/read  ->", doc.contents[0].text)

    msg = await client.get_prompt("explain", {"topic": "MCP"})          # user-controlled
    print("prompts/get     ->", msg.messages[0].content.text)

### 5. Build once, integrate everywhere
Two completely different “hosts” reuse the *same* server without either knowing its tools in advance: a dashboard that just lists capabilities, and a tiny keyword “agent” that picks a tool by reading descriptions. This is the N + M argument in code.

In [ ]:
async def dashboard_host(srv):
    async with Client(srv) as c:
        names = [t.name for t in (await c.list_tools()).tools]
        return f"dashboard: server '{c.server_info.name}' offers {names}"


async def keyword_agent_host(srv, request: str):
    async with Client(srv) as c:
        tools = (await c.list_tools()).tools
        # a stand-in for an LLM: pick the tool whose description shares the most words with the request
        score = lambda t: len(set(request.lower().split()) & set(t.description.lower().split()))
        best = max(tools, key=score)
        result = await c.call_tool(best.name, {"query": request.split()[-1]})
        return f"agent: chose {best.name!r} -> {result.structured_content}"


print(await dashboard_host(server))
print(await keyword_agent_host(server, "find the notes that contain lsp"))

### 6. Peek at the wire: raw JSON-RPC over stdio
Under the SDK, MCP is just JSON-RPC messages. Here the server runs as a real subprocess (the **stdio** transport) and we speak to it by writing one JSON object per line — no SDK on the client side. Every request carries the 2026-07-28 envelope in `_meta`.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "notes_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("notes")

    @server.tool()
    def search_notes(query: str) -> list[str]:
        """Return ids of notes containing the query."""
        return [n for n in ["mcp", "lsp"] if query.lower() in n]

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "notes_server.py")],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def rpc(id_, method, params):
    message = {"jsonrpc": "2.0", "id": id_, "method": method, "params": params}
    print(">>>", json.dumps(message))
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    print("<<<", json.dumps(reply, indent=2)[:700], "\n")
    return reply


rpc(1, "server/discover", {"_meta": META})
rpc(2, "tools/call", {"name": "search_notes", "arguments": {"query": "mc"}, "_meta": META})
proc.stdin.close()
exit_code = proc.wait(timeout=10)   # closing stdin is how a stdio client shuts a server down

Read the replies: the same `id` comes back, `resultType` is `"complete"`, and the server identifies itself in `_meta` (`io.modelcontextprotocol/serverInfo`). The `structuredContent` and the text `content` carry the same answer — one for programs, one for the model.

### 7. Optional: let a real model choose the tool
If `ANTHROPIC_API_KEY` is set, Claude receives the MCP tool's name, description and schema — exactly what the server advertised — and decides whether to call it. Otherwise the cell explains what would happen.

In [ ]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    async with Client(server) as client:
        tools = [{"name": t.name, "description": t.description, "input_schema": t.input_schema}
                 for t in (await client.list_tools()).tools]
        response = llm.messages.create(
            model="claude-opus-5-5", max_tokens=16000, tools=tools,
            messages=[{"role": "user", "content": "Which of my notes mention LSP?"}])
        for block in response.content:
            if block.type == "tool_use":
                result = await client.call_tool(block.name, block.input)
                print(f"Claude called {block.name}({block.input}) -> {result.structured_content}")
            elif block.type == "text":
                print("Claude:", block.text)
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude pick `search_notes` from the MCP tool list.")

### Your turn
1. Add a second tool `add_note(note_id: str, text: str) -> str` to the server in section 2, re-run sections 2–5, and confirm both hosts discover it **without any change to their code**.
2. In section 6, remove `"_meta": META` from the `tools/call` request and read the error code. Then send `"io.modelcontextprotocol/protocolVersion": "2020-01-01"` instead. What does each error tell the client to do?
3. Using the table in the lesson, decide whether each of these should be a tool, a resource or a prompt, and why: *the company style guide*, *send a Slack message*, *“write a release note from this diff”*, *today's on-call schedule*.